In [16]:
import pymor                # This defines the name 'pymor'
from pymor.basic import * # This gets all the pyMOR tools
import matplotlib.pyplot as plt

print(f"pyMOR is ready! Version: {pymor.__version__}")

pyMOR is ready! Version: 2025.1.2


In [17]:
# We start by implementing the Brusselator FOM
# Here: domain, parameter structure, linear operators

import numpy as np

# 1. Define the Domain and Mesh
# We use a 1D line domain [0, 1], thats per default
domain = LineDomain(left='neumann', right='dirichlet')

# Discretize the domain to get a Grid and Boundary Information
# 'diameter' controls the mesh fineness (h)
grid, bound_info = discretize_domain_default(LineDomain(), diameter=1/100) # grid contains the domain, bound_info the BCs

# 2. Define the Parameter structure
# These match the constants in your Brusselator equations
parameters = Parameters(alpha=1, beta=1, nu=1) # 1 because we have only scalars

# 3. Create the Linear Operators for the SHIFTED variables
# Since u and v are 0 at the right boundary, we set up a scalar
# problem that naturally results in a homogeneous Dirichlet system.

scalar_problem = StationaryProblem(   # time-independent 
    domain=domain,
    diffusion=ConstantFunction(1.0, 1),
    # In pyMOR, if you don't specify dirichlet_data, it defaults to 0!
    rhs=ConstantFunction(0.0, 1) 
)
scalar_fom, _ = discretize_stationary_cg(scalar_problem, diameter=1/100)

M_scalar = scalar_fom.operator # Mass matrix (needed for l2-inner product)
L_scalar = scalar_fom.operator # Stiffness matrix (needed for y_xx, z_xx)

print(f"Grid setup complete. Number of nodes: {grid.size(grid.dim)}")
print(f"Scalar operator size: {M_scalar.source.dim} x {M_scalar.range.dim}")

Accordion(children=(HTML(value='', layout=Layout(height='16em', width='100%')),), titles=('Log Output',))

Grid setup complete. Number of nodes: 101
Scalar operator size: 101 x 101


In [18]:
# Since we want a vector containing (u,v) as solution:

from pymor.operators.block import BlockDiagonalOperator

# 1. Create the Block Operators
# This puts L_scalar on the diagonal. The top-left handles 'u', 
# the bottom-right handles 'v'. They don't 'talk' to each other yet.
L = BlockDiagonalOperator([L_scalar, L_scalar])
M = BlockDiagonalOperator([M_scalar, M_scalar])

# 2. Define the Vector Space for the whole system
# This represents the combined space of [u, v]
system_space = L.source   # the vector space everything happens in

print(f"Scalar DoFs: {L_scalar.source.dim}")
print(f"Total System DoFs: {system_space.dim}")

Scalar DoFs: 101
Total System DoFs: 202


In [20]:
# Define the nonlinear operator

from pymor.operators.numpy import NumpyGenericOperator # We want to define any operator

def brusselator_reaction(U, mu): # U our vector array, mu contains current parameters
    
    alpha = mu['alpha'][0]
    beta = mu['beta'][0]
    
    u_data = U.to_numpy()   
    mid = u_data.shape[1] // 2   # since we stored u and v together
    u = u_data[:, :mid]
    v = u_data[:, mid:]

    # Lifted version
    res_u = (beta - 1) * u + (alpha**2) * v + (beta / alpha) * (u**2) + 2 * alpha * u * v + (u**2) * v
    res_v = -beta * u - (alpha**2) * v - (beta / alpha) * (u**2) - 2 * alpha * u * v - (u**2) * v
    
    return system_space.from_numpy(np.hstack([res_u, res_v]))

nonlinear_operator = NumpyGenericOperator(
    brusselator_reaction,
    dim_source=system_space.dim,  # Integer dimension
    dim_range=system_space.dim,   # Integer dimension
    parameters=parameters,
    name='reaction'
)


In [34]:
from pymor.operators.interface import Operator
import numpy as np

class BrusselatorReactionOperator(Operator):
    def __init__(self, space, parameters):
        self.source = space
        self.range = space
        self.parameters = parameters
        self.space = space
        self.linear = False  # Added this attribute to indicate the operator is nonlinear
        
    def apply(self, U, mu=None):
        # The same math logic we've used before
        alpha = mu['alpha'][0]
        beta = mu['beta'][0]
        
        u_data = U.to_numpy()
        mid = u_data.shape[1] // 2
        u = u_data[:, :mid]
        v = u_data[:, mid:]
        
        res_u = (beta - 1) * u + (alpha**2) * v + (beta / alpha) * (u**2) + 2 * alpha * u * v + (u**2) * v
        res_v = -beta * u - (alpha**2) * v - (beta / alpha) * (u**2) - 2 * alpha * u * v - (u**2) * v
        
        return self.range.from_numpy(np.hstack([res_u, res_v]))

# 1. Instantiate our custom operator using the EXACT space from L
nonlinear_operator = BrusselatorReactionOperator(L.source, parameters)

# 2. Combine them. Now pyMOR will see L.source == nonlinear_operator.source
full_operator = L + nonlinear_operator

# 3. Assemble the Model
from pymor.models.basic import InstationaryModel
from pymor.algorithms.timestepping import ImplicitEulerTimeStepper

stepper = ImplicitEulerTimeStepper(nt=200)

fom = InstationaryModel(
    T=20.0,
    initial_data=initial_data,
    operator=full_operator,
    rhs=None,
    mass=M,
    time_stepper=stepper,
    name='Brusselator_FOM'
)

print("Full Order Model successfully assembled!")

Full Order Model successfully assembled!


In [36]:
fom

InstationaryModel(
    20.0,
    VectorOperator(
        BlockVectorArray(
            BlockVectorSpace((NumpyVectorSpace(101), NumpyVectorSpace(101))),
            BlockVectorArrayImpl('??', BlockVectorSpace((NumpyVectorSpace(101), NumpyVectorSpace(101)))),
            _len=1),
        name='initial_data'),
    LincombOperator(
        (BlockDiagonalOperator(
             array([[NumpyMatrixOperator(<101x101 sparse, 300 nnz>),
                     ZeroOperator(NumpyVectorSpace(101), NumpyVectorSpace(101))],
                    [ZeroOperator(NumpyVectorSpace(101), NumpyVectorSpace(101)),
                     NumpyMatrixOperator(<101x101 sparse, 300 nnz>)]], dtype=object)),
         BrusselatorReactionOperator(
             BlockVectorSpace((NumpyVectorSpace(101), NumpyVectorSpace(101))),
             {alpha: 1, beta: 1, nu: 1})),
        (1.0, 1.0)),
    ZeroOperator(BlockVectorSpace((NumpyVectorSpace(101), NumpyVectorSpace(101))), NumpyVectorSpace(1)),
    mass=BlockDiagonalOperator(
